# Pretrained SAE Baseline and Steering Experiment


This notebook evaluates a pretrained sparse autoencoder (SAE) on GPT-2 Small. It examines SAE feature activations, tests Feature 974 ("mentions of Paris") on positive and negative examples, and performs a preliminary steering intervention.


In [ ]:
import scipy

print(scipy.__version__)
print(scipy.__file__)

In [ ]:
from transformer_lens import HookedTransformer
from sae_lens import SAE



In [ ]:
model = HookedTransformer.from_pretrained("gpt2-small")

In [ ]:
text = "The Eiffel Tower is located in Paris."

tokens = model.to_tokens(text)

print(tokens)
print(model.to_str_tokens(text))

In [ ]:
logits, cache = model.run_with_cache(tokens)

print(logits.shape)

In [ ]:
layer_0_activations = cache["blocks.0.hook_resid_post"]

print(layer_0_activations.shape)

In [ ]:
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
    device="cpu"
)

In [ ]:
print(sae.cfg)

In [ ]:
from sae_lens import HookedSAETransformer

In [ ]:
sae_model = HookedSAETransformer.from_pretrained_no_processing(
    "gpt2-small",
    device="cpu"
)

In [ ]:
tokens = sae_model.to_tokens(text)

_, cache = sae_model.run_with_cache(tokens)

layer_8_activations = cache["blocks.8.hook_resid_pre"]

print(layer_8_activations.shape)

In [ ]:
feature_activations = sae.encode(layer_8_activations)

print(feature_activations.shape)

In [ ]:
active_features = (feature_activations > 0).sum(dim=-1)

print(active_features)

In [ ]:
str_tokens = sae_model.to_str_tokens(text)

for token, count in zip(str_tokens, active_features[0]):
    print(f"{token!r}: {count.item()} active features")

In [ ]:
paris_activations = feature_activations[0, 9]

top_values, top_indices = paris_activations.topk(10)

for feature_id, value in zip(top_indices, top_values):
    print(f"Feature {feature_id.item()}: {value.item():.4f}")

In [ ]:
test_text = "I traveled to Paris last summer."

test_tokens = sae_model.to_tokens(test_text)

_, test_cache = sae_model.run_with_cache(test_tokens)

test_layer_8 = test_cache["blocks.8.hook_resid_pre"]

test_features = sae.encode(test_layer_8)

print(sae_model.to_str_tokens(test_text))

In [ ]:
paris_feature_value = test_features[0, 4, 974]

print(paris_feature_value.item())


In [ ]:
negative_text = "I traveled to London last summer."

negative_tokens = sae_model.to_tokens(negative_text)

_, negative_cache = sae_model.run_with_cache(negative_tokens)

negative_layer_8 = negative_cache["blocks.8.hook_resid_pre"]

negative_features = sae.encode(negative_layer_8)

print(sae_model.to_str_tokens(negative_text))

In [ ]:
london_feature_value = negative_features[0, 4, 974]

print(london_feature_value.item())

In [ ]:
def get_feature_activation(text, feature_id):
    tokens = sae_model.to_tokens(text)

    _, cache = sae_model.run_with_cache(tokens)

    layer_8 = cache["blocks.8.hook_resid_pre"]

    features = sae.encode(layer_8)

    return features[0, 1:, feature_id].max().item()

In [ ]:
print(get_feature_activation("I traveled to Paris last summer.", 974))
print(get_feature_activation("I traveled to London last summer.", 974))

In [ ]:
positive_examples = [
    "I traveled to Paris last summer.",
    "Paris is the capital of France.",
    "She moved to Paris for work.",
    "The conference will be held in Paris.",
    "We visited Paris during our vacation."
]

negative_examples = [
    "I traveled to London last summer.",
    "Berlin is the capital of Germany.",
    "She moved to Chicago for work.",
    "The conference will be held in Tokyo.",
    "We visited Rome during our vacation."
]

print("Positive examples:", len(positive_examples))
print("Negative examples:", len(negative_examples))

In [ ]:
positive_scores = []

for sentence in positive_examples:
    score = get_feature_activation(sentence, 974)
    positive_scores.append(score)
    print(f"{sentence} → {score:.2f}")

In [ ]:
negative_scores = []

for sentence in negative_examples:
    score = get_feature_activation(sentence, 974)
    negative_scores.append(score)
    print(f"{sentence} → {score:.2f}")

In [ ]:
import numpy as np

positive_mean = np.mean(positive_scores)
negative_mean = np.mean(negative_scores)

print(f"Average positive activation: {positive_mean:.2f}")
print(f"Average negative activation: {negative_mean:.2f}")

In [ ]:
import matplotlib.pyplot as plt

labels = ["Paris examples", "Non-Paris examples"]
means = [positive_mean, negative_mean]

plt.bar(labels, means)

plt.ylabel("Mean Feature 974 Activation")
plt.title("Feature 974: Paris vs Non-Paris Examples")
plt.savefig(
    "../results/figures/feature_974_paris_vs_nonparis.svg",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

In [ ]:
import pandas as pd

In [ ]:
results_df = pd.DataFrame({
    "sentence": positive_examples + negative_examples,
    "label": ["Paris"] * len(positive_examples) + ["Non-Paris"] * len(negative_examples),
    "feature_974_activation": positive_scores + negative_scores
})

results_df

In [ ]:
results_df.to_csv(
    "../results/tables/feature_974_detection_results.csv",
    index=False
)

print("Results saved successfully.")

In [ ]:
prompt = "The best place to spend my vacation is"

baseline_output = sae_model.generate(
    prompt,
    max_new_tokens=30,
    temperature=0
)

print(baseline_output)

In [ ]:
feature_id = 974
steering_strength = 20.0

feature_direction = sae.W_dec[feature_id]

def steering_hook(activation, hook):
    activation[:, -1, :] += steering_strength * feature_direction
    return activation

In [ ]:
with sae_model.hooks(
    fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]
):
    steered_output = sae_model.generate(
        prompt,
        max_new_tokens=30,
        temperature=0
    )

print(steered_output)


In [ ]:
steering_strength = 50.0

with sae_model.hooks(
    fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]
):
    steered_output_50 = sae_model.generate(
        prompt,
        max_new_tokens=30,
        temperature=0
    )

print(steered_output_50)

In [ ]:
steering_strength = 100.0

with sae_model.hooks(
    fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]
):
    steered_output_100 = sae_model.generate(
        prompt,
        max_new_tokens=30,
        temperature=0
    )

print(steered_output_100)

In [ ]:
steering_strength = 75.0

with sae_model.hooks(
    fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]
):
    steered_output_75 = sae_model.generate(
        prompt,
        max_new_tokens=30,
        temperature=0
    )

print(steered_output_75)

In [ ]:
steering_strength = 30.0

with sae_model.hooks(
    fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]
):
    steered_output_30 = sae_model.generate(
        prompt,
        max_new_tokens=30,
        temperature=0
    )

print(steered_output_30)

In [ ]:
steering_strength = 10.0

with sae_model.hooks(
    fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]
):
    steered_output_10 = sae_model.generate(
        prompt,
        max_new_tokens=30,
        temperature=0
    )

print(steered_output_10)

In [ ]:
steering_results = pd.DataFrame({
    "steering_strength": [0, 10, 20, 30, 50, 75, 100],
    "output": [
        baseline_output,
        steered_output_10,
        steered_output,
        steered_output_30,
        steered_output_50,
        steered_output_75,
        steered_output_100
    ]
})

steering_results

In [ ]:
steering_results.to_csv(
    "../results/tables/feature_974_steering_results.csv",
    index=False
)



In [ ]:
steering_results["paris_count"] = (
    steering_results["output"]
    .str.lower()
    .str.count("paris")
)

steering_results[
    ["steering_strength", "mentions_paris", "paris_count"]
]

In [ ]:
steering_results.to_csv(
    "../results/tables/feature_974_steering_results.csv",
    index=False
)

